Copyright **`(c)`** 2022 Giovanni Squillero `<squillero@polito.it>`  
[`https://github.com/squillero/computational-intelligence`](https://github.com/squillero/computational-intelligence)  
Free for personal or classroom use; see [`LICENSE.md`](https://github.com/squillero/computational-intelligence/blob/master/LICENSE.md) for details.  


# Lab 1: Set Covering

First lab + peer review. List this activity in your final report, it will be part of your exam.

## Task

Given a number $N$ and some lists of integers $P = (L_0, L_1, L_2, ..., L_n)$, 
determine, if possible, $S = (L_{s_0}, L_{s_1}, L_{s_2}, ..., L_{s_n})$
such that each number between $0$ and $N-1$ appears in at least one list

$$\forall n \in [0, N-1] \ \exists i : n \in L_{s_i}$$

and that the total numbers of elements in all $L_{s_i}$ is minimum. 

## Instructions

* Create the directory `lab1` inside the course repo (the one you registered with Andrea)
* Put a `README.md` and your solution (all the files, code and auxiliary data if needed)
* Use `problem` to generate the problems with different $N$
* In the `README.md`, report the the total numbers of elements in $L_{s_i}$ for problem with $N \in [5, 10, 20, 100, 500, 1000]$ and the total number on $nodes$ visited during the search. Use `seed=42`.
* Use `GitHub Issues` to peer review others' lab

## Notes

* Working in group is not only allowed, but recommended (see: [Ubuntu](https://en.wikipedia.org/wiki/Ubuntu_philosophy) and [Cooperative Learning](https://files.eric.ed.gov/fulltext/EJ1096789.pdf)). Collaborations must be explicitly declared in the `README.md`.
* [Yanking](https://www.emacswiki.org/emacs/KillingAndYanking) from the internet is allowed, but sources must be explicitly declared in the `README.md`.

**Deadline**

* Sunday, October 16th 23:59:59 for the working solution
* Sunday, October 23rd 23:59:59 for the peer reviews

In [10]:
import random

In [31]:
def problem(N, seed=None):
    random.seed(seed)
    return [
        list(set(random.randint(0, N - 1) for n in range(random.randint(N // 5, N // 2))))
        for n in range(random.randint(N, N * 5))
    ]

In [543]:
import logging


def greedy(N):
    goal = set(range(N))
    covered = set()
    solution = list()
    all_lists = sorted(problem(N, seed=42), key=lambda l: len(l))
    while goal != covered:
        x = all_lists.pop(0)
        if not set(x) < covered:
            solution.append(x)
            covered |= set(x)

    logging.info(
        f"Greedy solution for N={N}: w={sum(len(_) for _ in solution)} (bloat={(sum(len(_) for _ in solution)-N)/N*100:.0f}%)"
    )
    logging.debug(f"{solution}")

In [544]:
logging.getLogger().setLevel(logging.INFO)
for N in [5, 10, 20, 100, 500, 1000]:
    greedy(N)

INFO:root:Greedy solution for N=5: w=5 (bloat=0%)
INFO:root:Greedy solution for N=10: w=13 (bloat=30%)
INFO:root:Greedy solution for N=20: w=46 (bloat=130%)
INFO:root:Greedy solution for N=100: w=332 (bloat=232%)
INFO:root:Greedy solution for N=500: w=2162 (bloat=332%)
INFO:root:Greedy solution for N=1000: w=4652 (bloat=365%)


In [545]:
%timeit greedy(1_000)

INFO:root:Greedy solution for N=1000: w=4652 (bloat=365%)
INFO:root:Greedy solution for N=1000: w=4652 (bloat=365%)
INFO:root:Greedy solution for N=1000: w=4652 (bloat=365%)
INFO:root:Greedy solution for N=1000: w=4652 (bloat=365%)
INFO:root:Greedy solution for N=1000: w=4652 (bloat=365%)
INFO:root:Greedy solution for N=1000: w=4652 (bloat=365%)
INFO:root:Greedy solution for N=1000: w=4652 (bloat=365%)
INFO:root:Greedy solution for N=1000: w=4652 (bloat=365%)


669 ms ± 3.19 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [548]:
import logging
import copy

def h(sol, current_x):
    common_elements = len(set(sol) & set(current_x))
    new_elements = len(current_x) - common_elements

    if (new_elements == 0):
        return float('inf')

    res = common_elements/new_elements
    return res

def my_greedy(N):
    goal = set(range(N))

    lists = sorted(problem(N, seed=42), key=lambda l: -len(l))

    starting_x = lists.pop(0)
    
    sol = list()
    sol.append(starting_x)
    
    flat_sol = list(starting_x)

    covered = set(starting_x)
    while goal != covered:
        most_promising_x = min(lists, key = lambda x: h(flat_sol, x))
        lists.remove(most_promising_x)
        
        flat_sol.extend(most_promising_x)
        sol.append(most_promising_x)

        covered |= set(most_promising_x)

    w = len(flat_sol)

    logging.info(
        f"Greedy solution for N={N}: w={w} (bloat={(w-N)/N*100:.0f}%)"
    )
    logging.debug(f"{sol}")
    return sol

In [549]:
logging.getLogger().setLevel(logging.INFO)

approx_results = dict()

logging.getLogger().setLevel(logging.INFO)
for N in [5, 10, 20, 100, 500, 1000]:
    my_greedy(N)


INFO:root:Greedy solution for N=5: w=5 (bloat=0%)
INFO:root:Greedy solution for N=10: w=10 (bloat=0%)
INFO:root:Greedy solution for N=20: w=24 (bloat=20%)
INFO:root:Greedy solution for N=100: w=182 (bloat=82%)
INFO:root:Greedy solution for N=500: w=1262 (bloat=152%)
INFO:root:Greedy solution for N=1000: w=2878 (bloat=188%)


In [468]:
lists = sorted(problem(10, seed=42), key=lambda l: -len(l))

print(lists)
print(len(lists))

[[0, 3, 4, 7, 9], [3, 4, 5, 6, 8], [0, 1, 3, 4, 5], [0, 9, 4, 5], [1, 3, 4, 9], [1, 3, 4, 6], [8, 2, 3, 7], [8, 0, 4, 1], [1, 4, 5, 6], [8, 9, 3, 6], [8, 1, 3, 7], [8, 2, 3, 7], [1, 3, 6, 7], [1, 2, 3], [8, 9, 3], [4, 5, 6], [1, 3, 5], [8, 1, 6], [9, 3, 5], [1, 3, 6], [2, 5, 7], [8, 2, 3], [3, 6, 7], [2, 3, 4], [9, 2, 6], [0, 4, 7], [8, 1, 4], [8, 2, 7], [4, 5, 6], [0, 9, 3], [0, 4], [9, 6], [0, 1], [8, 3], [1, 6], [0, 3], [0, 3], [9, 6], [0, 1], [2, 5], [9, 5], [9, 3], [1, 7], [8, 2], [0, 5], [0, 3], [0, 5], [8, 3], [5, 6], [6]]
50
